In [1]:
"""
Imports
"""
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn import CrossEntropyLoss

In [2]:
"""
Load Dataset
"""
df = pd.read_csv("/content/dataset.csv", sep=";")

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Train / Test split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['Label']
)

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

Train samples: 136732
Test samples: 34183


In [3]:
"""
Label Encoding
"""
labels = sorted(df['Label'].unique())
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

print("Label mapping:", label_to_idx)

Label mapping: {'Anthropic': 0, 'Google': 1, 'Human': 2, 'Meta': 3, 'OpenAI': 4}


In [4]:
"""
Tokenizer + Model
"""
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Using device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cuda


In [5]:
"""
Dataset Class
"""
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, label_to_idx, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.label_to_idx = label_to_idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.label_to_idx[self.labels[idx]]

        encoding = tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


In [6]:
"""
Datasets + Loaders
"""
train_dataset = TransformerDataset(
    train_df['Text'].values,
    train_df['Label'].values,
    tokenizer,
    label_to_idx
)

test_dataset = TransformerDataset(
    test_df['Text'].values,
    test_df['Label'].values,
    tokenizer,
    label_to_idx
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 8546
Test batches: 2137


In [ ]:
"""
Loss + Optimizer
"""
# Class weights
class_counts = train_df['Label'].value_counts().sort_index().values
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
weights = weights / weights.sum()

criterion = CrossEntropyLoss(weight=weights.to(device))
optimizer = AdamW(model.parameters(), lr=2e-5)

"""
Learning rate scheduler (help to reduce loss during the latest step of training)
"""
from transformers import get_linear_schedule_with_warmup

# Calculate total training steps
epochs = 5
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps), # 10% warmup
    num_training_steps=total_steps
)

In [8]:
"""
Training Function
"""
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            #labels=labels
        )

        #loss = outputs.loss
        logits = outputs.logits
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()
        scheduler.step() # learning rate scheduler

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total


In [9]:

"""
Validation Function
"""
def validate(model, dataloader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(dataloader), 100 * correct / total


In [10]:
"""
Training Loop with Early Stopping
"""
best_loss = float('inf')
patience = 2
patience_counter = 0
#epochs = 5

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
        print("✓ Best model saved")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print("Early stopping triggered")
            break



Epoch 1/5
Train Loss: 0.6069 | Train Acc: 73.21%
Val Loss: 0.5439 | Val Acc: 79.87%
✓ Best model saved

Epoch 2/5
Train Loss: 0.3048 | Train Acc: 87.90%
Val Loss: 0.3838 | Val Acc: 86.31%
✓ Best model saved

Epoch 3/5
Train Loss: 0.1904 | Train Acc: 92.75%
Val Loss: 0.4074 | Val Acc: 86.17%
No improvement (1/2)

Epoch 4/5
Train Loss: 0.1148 | Train Acc: 95.83%
Val Loss: 0.5087 | Val Acc: 85.64%
No improvement (2/2)
Early stopping triggered


In [11]:
"""
Load Best Model
"""
model.load_state_dict(torch.load("/content/best_model.pth"))


<All keys matched successfully>

In [12]:
"""
Prediction Function
"""
def predict(text, model, tokenizer, device, idx_to_label):
    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()

    return idx_to_label[pred], confidence

In [13]:
"""
Evaluate on exemplos dataset
"""
exemplos_df = pd.read_csv("/content/dataset-exemplos.csv", sep=";")

real = exemplos_df["Label"].tolist()
preds = []

for text in exemplos_df["Text"]:
    p, _ = predict(text, model, tokenizer, device, idx_to_label)
    preds.append(p)

accuracy = sum(1 for r, p in zip(real, preds) if r == p) / len(real)
print(f"Accuracy (exemplos): {accuracy:.4f}")


Accuracy (exemplos): 0.5120


In [14]:
"""
Evaluate on subm1 revealed dataset
"""
revealed_df = pd.read_csv("/content/subm1_labels_revealed.csv", sep=";")

real = revealed_df["Label"].tolist()
preds = []

for text in revealed_df["Text"]:
    p, _ = predict(text, model, tokenizer, device, idx_to_label)
    preds.append(p)

accuracy = sum(1 for r, p in zip(real, preds) if r == p) / len(real)
print(f"Accuracy (exemplos): {accuracy:.4f}")


Accuracy (exemplos): 0.4900


In [15]:
"""
Generate Submission
"""
subm_df = pd.read_csv("/content/subm2.csv", sep=";")

results = []
for text in subm_df["Text"]:
    p, _ = predict(text, model, tokenizer, device, idx_to_label)
    results.append(p)

subm_df["Label"] = results
subm_df = subm_df.drop(columns=["Text"])
subm_df.to_csv("/content/subm2-g1-MMC-B.csv", sep=";", index=False)

print("Submission file saved!")

Submission file saved!
